In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace

# ================================
# 1. READ BRONZE TABLE
# ================================

df = spark.table("electronics_retailer_clg.bronze.products")


# ================================
# 2. CLEAN COLUMN NAMES
# ================================

df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


# ================================
# 3. TRIM SPACES
# ================================

for c in df.columns:
    df = df.withColumn(c, trim(col(c)))


# ================================
# 4. CLEAN PRICE COLUMN (FIX ERROR 🔥)
# ================================

# Remove $ symbol
df = df.withColumn(
    "unit_price_usd",
    regexp_replace("unit_price_usd", "\\$", "")
)

# Remove commas
df = df.withColumn(
    "unit_price_usd",
    regexp_replace("unit_price_usd", ",", "")
)


# ================================
# 5. FIX DATA TYPES
# ================================

df = df.withColumn("productkey", col("productkey").cast("int")) \
       .withColumn("unit_price_usd", col("unit_price_usd").cast("double"))


# ================================
# 6. KEEP ONLY REQUIRED COLUMNS
# ================================

df = df.select(
    "productkey",
    "category",
    "unit_price_usd"
)


# ================================
# 7. SHOW TABLE (IMPORTANT)
# ================================

display(df)


# ================================
# 8. WRITE TO SILVER
# ================================

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronics_retailer_clg.silver.products")

print("✅ Products cleaned successfully (price issue fixed)")